In [ ]:
print("Working") # sometimes stuff takes a long time to actually run so this is to see if its actually running or notw


Working


Import stuff

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


Combining all of the GaN datasets into one big dataframe.


In [ ]:
files = [f"GaN{year}.csv" for year in range(2006, 2025)] #All of the files are named GaN(year).csv so this just gets them looping from the years of 2006 to 2025, and adding GaN and .csv

all_dataframes = []
for f in files:
    try:
        df = pd.read_csv(f, engine='python')
        all_dataframes.append(df)
    except pd.errors.ParserError as e:
        print(f"Warning: Could not read file {f} due to ParserError: {e}. Skipping this file.")
    except FileNotFoundError:
        print(f"Warning: File {f} not found. Skipping this file.")

if all_dataframes:
    star_data = pd.concat(all_dataframes, axis=0) # Changed axis to 0 for row-wise concatenation
else:
    print("No valid dataframes were read. 'star_data' will not be created.")
    star_data = pd.DataFrame() # Initialize as empty DataFrame if no valid data

No valid dataframes were read. 'star_data' will not be created.


Take a look at the data!

In [ ]:
star_data.head()
star_data.tail()

""


Clean the data (the text after dashes or brackets)


In [ ]:
#GaN data
if not star_data.empty and "Country" in star_data.columns:
    star_data["Country"] = (
        star_data["Country"]
        .str.split("-", n=1)
        .str[0]
        .str.strip()
    )

In [ ]:
star_data.head()
star_data.tail()

""


Now for the HDI dataset.

Put the data into a pandas dataframe (its an xlsx file)

In [ ]:
hdi=pd.read_excel('HDI.xlsx')

FileNotFoundError: [Errno 2] No such file or directory: 'HDI.xlsx'

Look at it


In [ ]:
hdi.head()

Clean the data.


Remove a glitched column, and stuff in brackets or stuff after dashes

In [ ]:
hdi = hdi.drop(columns=["mmm nmmmmmnnnnmmmmm"]) #not sure why but theres a column named this, i have to remove it, its COMPLETELY empty


#HDI data
hdi["Country"] = (
    hdi["Country"]
    .str.replace(r"\s*-\s*.*$", "", regex=True)
    .str.replace(r"\s*\(.*?\)", "", regex=True)
    .str.strip()
)



There are a LOT of countries and territories that either arent in the HDI, or are under a different name. So, all of those countries had to be hard-coded for to combine the datasets properly.

In [ ]:
#Countries that have a different name
gan_to_hdi = {
    "Turkey":                                    "Türkiye",
    "Vietnam":                                   "Viet Nam",
    "Russia":                                    "Russian Federation",
    "Macedonia":                                 "North Macedonia",
    "Czech Republic":                            "Czechia",
    "South Korea":                               "Korea",          # HDI: "Korea (Republic of)"
    "North Korea":                               "Korea",          # HDI: "Korea (Democratic People's Rep. of)"
    "The Bahamas":                               "Bahamas",
    "Democratic Republic of the Congo":          "Congo",          # HDI: "Congo (Democratic Republic of the)"
    "Syria":                                     "Syrian Arab Republic",
    "Laos":                                      "Lao People's Democratic Republic",
    "St Vincent and the Grenadines":             "Saint Vincent and the Grenadines",
    "The Gambia":                                "Gambia",
    "Cape Verde":                                "Cabo Verde",
    "St Kitts and Nevis":                        "Saint Kitts and Nevis",
}

#Territories - these will be excluded because the HDI does not calculate these.
territories_to_exclude = {
    "Puerto Rico",
    "Cayman Islands",
    "Taiwan",
    "Greenland",
    "Isle of Man",
    "Virgin Islands",
    "Bermuda",
    "Guam",
    "Martinique",
    "Guernsey",
    "Aruba",
    "New Caledonia",
    "Antarctica",
    "French Polynesia",
    "Reunion",
    "British Indian Ocean Territory",
    "British Virgin Islands",
    "Gibraltar",
    "Cook Islands",
    "Turks and Caicos Islands",
    "Norfolk Island",
    "Baker Island",
    "Pitcairn Islands",
    "Netherlands Antilles",
    "Guadeloupe",
    "Falkland Islands",
    "Wake Island",
    "South Georgia and the South Sandwich Islands",
    "Northern Mariana Islands",
}

if not star_data.empty and "Country" in star_data.columns:
    #Put the renamed ones into the datasets
    star_data["Country"] = star_data["Country"].replace(gan_to_hdi)

    #Delete the territories o7
    star_data = star_data[~star_data["Country"].isin(territories_to_exclude)].reset_index(drop=True)

    # Removed the specific filter for 'United States' to retain all valid countries
else:
    print("Warning: star_data is empty or 'Country' column is missing. Skipping country-specific filtering and renaming.")

Take a look at the cleaned up data


In [ ]:
hdi.head()

Merge them

In [ ]:
hdi.columns = hdi.columns.str.strip()

if "Value" not in star_data.columns: #You get multiple columns if you accidentally run this block twice, so this double checks
    star_data = star_data.merge(
        hdi[["Country", "Value"]],
        on="Country",
        how="left"
    )



Take a look at our cleaned and combined data!

In [ ]:
star_data.head()
star_data.tail()

Debug stuff, so I can look at the dataset and make sure everythings going right. You dont need to look at this unless you want to see the data.


In [ ]:
import random

filename = f"star_data_with_hdi_{random.randint(100000, 999999)}.csv" #make a csv file output with a random number so there is no output
star_data.to_csv(filename, index=False)

print("Saved as:", filename)


More preprocessing needs to be done

In [ ]:
sns.barplot(x=star_data['Value'],y=star_data['LimitingMag'],data=star_data)


Now let's try doing it with all of the countries! The first thing I want to try and do is try to average all the radiances of each country. We have done model development, however doing more simple linear and logarithmic regression mapped would be more significant and helpful for the research topic.

In [ ]:
numerical_cols = star_data.select_dtypes(include=['number']).columns
if 'ID' in numerical_cols: # ID is an identifier, not a measurement to average
    numerical_cols = numerical_cols.drop('ID')

# Group by 'Country' and calculate the mean for numerical columns
avg_star_data_by_country = star_data.groupby('Country')[numerical_cols].mean().reset_index()

print("Averaged data by country:")
print(avg_star_data_by_country.head())

In [ ]:
# First, filter the original star_data to only include valid LimitingMag values (0-10)
star_data_valid_mag = star_data[(star_data['LimitingMag'] >= 0) & (star_data['LimitingMag'] <= 10)].copy()

# Identify numerical columns from this filtered dataframe for averaging
numerical_cols_valid_mag = star_data_valid_mag.select_dtypes(include=['number']).columns
if 'ID' in numerical_cols_valid_mag: # ID is an identifier, not a measurement to average
    numerical_cols_valid_mag = numerical_cols_valid_mag.drop('ID')

# Group by 'Country' and calculate the mean for these numerical columns
avg_star_data_by_country_recalculated = star_data_valid_mag.groupby('Country')[numerical_cols_valid_mag].mean().reset_index()

print("Recalculated averaged data by country (ignoring invalid LimitingMag values):")
display(avg_star_data_by_country_recalculated.head())

In [ ]:
# Filter out 'LimitingMag' values that are below 0 or above 10 so we dont get those cursed values.
avg_star_data_by_country_filtered = avg_star_data_by_country[(avg_star_data_by_country['LimitingMag'] >= 0) & (avg_star_data_by_country['LimitingMag'] <= 10)]

print("Averaged data by country after filtering LimitingMag:")
display(avg_star_data_by_country_filtered.head())

In [ ]:
# Calculate total data points per country from the original star_data
total_data_points_per_country = star_data.groupby('Country').size().reset_index(name='total_data_points')

# Calculate valid data points per country from the filtered star_data_valid_mag
valid_data_points_per_country = star_data_valid_mag.groupby('Country').size().reset_index(name='valid_data_points')

# Merge total data points into the averaged DataFrame
avg_star_data_by_country_recalculated = avg_star_data_by_country_recalculated.merge(
    total_data_points_per_country, on='Country', how='left'
)

# Merge valid data points to calculate dropped ones
avg_star_data_by_country_recalculated = avg_star_data_by_country_recalculated.merge(
    valid_data_points_per_country, on='Country', how='left'
)

# Calculate dropped data points
avg_star_data_by_country_recalculated['dropped_data_points'] = \
    avg_star_data_by_country_recalculated['total_data_points'] - avg_star_data_by_country_recalculated['valid_data_points']

# Fill any NaN in 'valid_data_points' or 'dropped_data_points' with 0 (for countries with no valid data)
avg_star_data_by_country_recalculated['valid_data_points'] = avg_star_data_by_country_recalculated['valid_data_points'].fillna(0).astype(int)
avg_star_data_by_country_recalculated['dropped_data_points'] = avg_star_data_by_country_recalculated['dropped_data_points'].fillna(avg_star_data_by_country_recalculated['total_data_points']).astype(int)

print("Averaged data with total and dropped data points:")
display(avg_star_data_by_country_recalculated.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure 'Value' column is present in avg_star_data_by_country_recalculated
# This step is added to address the ValueError: 'Value' column not found.
if 'Value' not in avg_star_data_by_country_recalculated.columns:
    avg_star_data_by_country_recalculated = avg_star_data_by_country_recalculated.merge(
        hdi[['Country', 'Value']],
        on='Country',
        how='left'
    )

# Convert 'Value' column to numeric, coercing errors to NaN
avg_star_data_by_country_recalculated['Value'] = pd.to_numeric(avg_star_data_by_country_recalculated['Value'], errors='coerce')

# Drop rows where 'Value' or 'LimitingMag' are NaN (which could be a result of coercion)
# Also ensure 'valid_data_points' is not NaN if used for size/hue
plot_data = avg_star_data_by_country_recalculated.dropna(subset=['Value', 'LimitingMag', 'valid_data_points']).copy()

# Create a scatter plot of LimitingMag vs. Value, with marker size based on valid_data_points
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='Value',
    y='LimitingMag',
    size='valid_data_points',  # Use valid_data_points to determine marker size
    sizes=(20, 1000),         # Define the range of marker sizes
    hue='valid_data_points',  # Color points by valid_data_points for better visual differentiation
    data=plot_data,
    legend='brief'
)

plt.title('Average Limiting Magnitude vs. HDI Value by Country (Weighted by Data Points)')
plt.xlabel('HDI Value')
plt.ylabel('Average Limiting Magnitude')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assuming plot_data is already defined and cleaned from previous steps
# plot_data contains 'Value' (HDI) and 'LimitingMag' (average for country)

plt.figure(figsize=(12, 8))
sns.regplot(
    x='Value',
    y='LimitingMag',
    data=plot_data,
    scatter=False, # Only show the line, not the individual points
    ci=95,         # Show 95% confidence interval for the regression line
    line_kws={'color': 'red'} # Customize line color
)

plt.title('Average Limiting Magnitude vs. HDI Value by Country (Regression Line)')
plt.xlabel('HDI Value')
plt.ylabel('Average Limiting Magnitude')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

Now, lets try graphing the same thing BUT without averaging.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure 'Value' column is present in star_data_valid_mag
# This step merges the HDI 'Value' back into the filtered star_data
if 'Value' not in star_data_valid_mag.columns:
    star_data_valid_mag_with_hdi = star_data_valid_mag.merge(
        hdi[['Country', 'Value']],
        on='Country',
        how='left'
    )
else:
    star_data_valid_mag_with_hdi = star_data_valid_mag.copy()

# Convert 'Value' column to numeric, coercing errors to NaN
star_data_valid_mag_with_hdi['Value'] = pd.to_numeric(star_data_valid_mag_with_hdi['Value'], errors='coerce')

# Drop rows where 'Value' or 'LimitingMag' are NaN for plotting
plot_data_raw = star_data_valid_mag_with_hdi.dropna(subset=['Value', 'LimitingMag']).copy()

# Create a scatter plot of LimitingMag vs. Value with a smoothed regression line (LOWESS)
plt.figure(figsize=(12, 8))
sns.regplot(
    x='Value',
    y='LimitingMag',
    data=plot_data_raw,
    scatter_kws={'alpha': 0.1, 's': 10}, # Make points translucent and smaller for large datasets
    line_kws={'color': 'red'},          # Customize line color
    lowess=True                          # Use LOWESS for a smoothed, non-linear regression line
)

plt.title('Limiting Magnitude vs. HDI Value (All Data Points with LOWESS Smoothing)')
plt.xlabel('HDI Value')
plt.ylabel('Limiting Magnitude')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Logarithmic Regression (Averaged Data)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Assuming plot_data is already defined and cleaned from previous steps
# plot_data contains 'Value' (HDI) and 'LimitingMag' (average for country)

# Filter out any zero or negative 'Value' for logarithmic transformation
plot_data_log = plot_data[plot_data['Value'] > 0].copy()

plt.figure(figsize=(12, 8))
sns.regplot(
    x='Value',
    y='LimitingMag',
    data=plot_data_log,
    scatter_kws={'s': 50, 'alpha': 0.7},
    line_kws={'color': 'red'},
    logx=True, # Apply logarithmic transformation to the x-axis
    ci=95      # Show 95% confidence interval for the regression line
)

plt.title('Average Limiting Magnitude vs. HDI Value (Logarithmic Regression on X-axis)')
plt.xlabel('HDI Value (Log Scale)')
plt.ylabel('Average Limiting Magnitude')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Evaluating Regression Fit (R-squared)

To understand how well our models explain the relationship between HDI Value and Limiting Magnitude, we can calculate the R-squared value for both the linear and logarithmic regressions.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np

# --- Linear Regression R-squared (Averaged Data) ---

# Prepare data for linear regression
X_linear = plot_data[['Value']].values
y_linear = plot_data['LimitingMag'].values

# Create and fit the linear regression model
linear_model = LinearRegression()
linear_model.fit(X_linear, y_linear)

# Make predictions
y_linear_pred = linear_model.predict(X_linear)

# Calculate R-squared
r_squared_linear = r2_score(y_linear, y_linear_pred)

print(f"R-squared for Linear Regression (Averaged Data): {r_squared_linear:.4f}")

In [ ]:
# --- Logarithmic Regression R-squared (Averaged Data) ---

# Prepare data for logarithmic regression on X-axis
# Ensure 'Value' is greater than 0 before taking log
X_log = np.log(plot_data_log[['Value']]).values
y_log = plot_data_log['LimitingMag'].values

# Create and fit the linear regression model on the log-transformed x-axis
log_model = LinearRegression()
log_model.fit(X_log, y_log)

# Make predictions
y_log_pred = log_model.predict(X_log)

# Calculate R-squared
r_squared_log = r2_score(y_log, y_log_pred)

print(f"R-squared for Logarithmic Regression (Averaged Data): {r_squared_log:.4f}")